In [1]:
import requests
import pandas as pd
import time
import os

In [2]:
# Base coordinates: Jaipur, Rajasthan
BASE_LAT = 26.9124
BASE_LON = 75.7873

# 4 zones - slightly offset coordinates to simulate a large solar plant
ZONES = {
    "Zone_A": {"lat": BASE_LAT + 0.005, "lon": BASE_LON + 0.005},
    "Zone_B": {"lat": BASE_LAT + 0.005, "lon": BASE_LON - 0.005},
    "Zone_C": {"lat": BASE_LAT - 0.005, "lon": BASE_LON + 0.005},
    "Zone_D": {"lat": BASE_LAT - 0.005, "lon": BASE_LON - 0.005},
}

In [3]:
START_DATE = "2020-01-01"
END_DATE = "2024-12-31"

HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",        # GHI
    "direct_normal_irradiance",   # DNI
    "diffuse_radiation",          # DHI
    "global_tilted_irradiance",   # GTI
    "sunshine_duration",
]

OUTPUT_DIR = "./data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

In [4]:
def fetch_zone_data(zone_name, lat, lon, start_date, end_date):
    """Fetch historical hourly data for a single zone."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARS),
        "timezone": "Asia/Kolkata",
        "tilt": 24,        # typical fixed-tilt panel angle for ~27°N latitude
        "azimuth": 0,      # 0 = south-facing
    }

    print(f"Fetching data for {zone_name} ({lat}, {lon})...")
    response = requests.get(BASE_URL, params=params, timeout=60)

    if response.status_code != 200:
        print(f"  ERROR {response.status_code}: {response.text[:300]}")
        return None

    data = response.json()
    hourly = data.get("hourly", {})

    df = pd.DataFrame(hourly)
    df["zone"] = zone_name
    df["latitude"] = lat
    df["longitude"] = lon

    print(f"  -> {len(df)} rows fetched")
    return df

In [5]:
all_zone_dfs = []

for zone_name, coords in ZONES.items():
    df = fetch_zone_data(zone_name, coords["lat"], coords["lon"], START_DATE, END_DATE)
    if df is not None:
        all_zone_dfs.append(df)

        zone_path = os.path.join(OUTPUT_DIR, f"{zone_name}_weather.csv")
        df.to_csv(zone_path, index=False)
        print(f"  Saved -> {zone_path}")

    time.sleep(1)  # be polite to the free API

Fetching data for Zone_A (26.9174, 75.7923)...
  -> 43848 rows fetched
  Saved -> ./data\Zone_A_weather.csv
Fetching data for Zone_B (26.9174, 75.7823)...
  -> 43848 rows fetched
  Saved -> ./data\Zone_B_weather.csv
Fetching data for Zone_C (26.907400000000003, 75.7923)...
  -> 43848 rows fetched
  Saved -> ./data\Zone_C_weather.csv
Fetching data for Zone_D (26.907400000000003, 75.7823)...
  -> 43848 rows fetched
  Saved -> ./data\Zone_D_weather.csv


In [6]:
combined = pd.concat(all_zone_dfs, ignore_index=True)
combined_path = os.path.join(OUTPUT_DIR, "all_zones_weather_combined.csv")
combined.to_csv(combined_path, index=False)

print(f"Combined file: {combined_path}")
print(f"Total rows: {len(combined)}")
print(f"Columns: {list(combined.columns)}")
print(f"Approx file size: {os.path.getsize(combined_path) / (1024*1024):.2f} MB")

Combined file: ./data\all_zones_weather_combined.csv
Total rows: 175392
Columns: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_radiation', 'global_tilted_irradiance', 'sunshine_duration', 'zone', 'latitude', 'longitude']
Approx file size: 16.75 MB


In [7]:
combined

,time,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,shortwave_radiation,direct_normal_irradiance,diffuse_radiation,global_tilted_irradiance,sunshine_duration,zone,latitude,longitude
0,2020-01-01T00:00,6.4,94,0.0,967.0,51,9.7,63,0.0,0.0,0.0,0.0,0.0,Zone_A,26.9174,75.7923
1,2020-01-01T01:00,6.8,92,0.0,966.6,12,11.0,58,0.0,0.0,0.0,0.0,0.0,Zone_A,26.9174,75.7923
2,2020-01-01T02:00,6.7,92,0.0,966.2,10,10.9,56,0.0,0.0,0.0,0.0,0.0,Zone_A,26.9174,75.7923
3,2020-01-01T03:00,6.5,92,0.0,966.1,13,10.0,60,0.0,0.0,0.0,0.0,0.0,Zone_A,26.9174,75.7923
4,2020-01-01T04:00,6.4,93,0.0,966.3,16,9.3,62,0.0,0.0,0.0,0.0,0.0,Zone_A,26.9174,75.7923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175387,2024-12-31T19:00,11.3,85,0.0,969.7,2,4.5,76,0.0,0.0,0.0,0.0,0.0,Zone_D,26.9074,75.7823
175388,2024-12-31T20:00,10.7,88,0.0,970.1,31,4.7,67,0.0,0.0,0.0,0.0,0.0,Zone_D,26.9074,75.7823
175389,2024-12-31T21:00,10.1,91,0.0,970.0,11,5.3,62,0.0,0.0,0.0,0.0,0.0,Zone_D,26.9074,75.7823
175390,2024-12-31T22:00,9.4,94,0.0,970.3,1,4.9,54,0.0,0.0,0.0,0.0,0.0,Zone_D,26.9074,75.7823


In [8]:
# Load the combined dataset
combined = pd.read_csv("./data/all_zones_weather_combined.csv")

# Basic shape check
print("Shape:", combined.shape)
print()

# Null counts per column
print("Null counts per column:")
print(combined.isnull().sum())
print()

# Percentage of nulls per column (easier to read at scale)
print("Null percentage per column:")
print((combined.isnull().sum() / len(combined) * 100).round(2))

Shape: (175392, 16)

Null counts per column:
time                        0
temperature_2m              0
relative_humidity_2m        0
precipitation               0
surface_pressure            0
cloud_cover                 0
wind_speed_10m              0
wind_direction_10m          0
shortwave_radiation         0
direct_normal_irradiance    0
diffuse_radiation           0
global_tilted_irradiance    0
sunshine_duration           0
zone                        0
latitude                    0
longitude                   0
dtype: int64

Null percentage per column:
time                        0.0
temperature_2m              0.0
relative_humidity_2m        0.0
precipitation               0.0
surface_pressure            0.0
cloud_cover                 0.0
wind_speed_10m              0.0
wind_direction_10m          0.0
shortwave_radiation         0.0
direct_normal_irradiance    0.0
diffuse_radiation           0.0
global_tilted_irradiance    0.0
sunshine_duration           0.0
zone             

In [9]:
import pvlib
print(pvlib.__version__)

0.15.2


In [10]:
from pvlib.location import Location

# Build solar position lookup per zone (since each zone has slightly different lat/lon)
zenith_dfs = []

for zone_name, coords in ZONES.items():
    zone_subset = combined[combined["zone"] == zone_name].copy()
    
    # Convert the 'time' column to a proper datetime index
    zone_subset["time"] = pd.to_datetime(zone_subset["time"])
    zone_subset["time"] = zone_subset["time"].dt.tz_localize("Asia/Kolkata")
    
    loc = Location(latitude=coords["lat"], longitude=coords["lon"], tz="Asia/Kolkata")
    
    solpos = loc.get_solarposition(zone_subset["time"])
    
    zone_subset["solar_zenith"] = solpos["apparent_zenith"].values
    zone_subset["solar_elevation"] = solpos["apparent_elevation"].values
    
    zenith_dfs.append(zone_subset)
    print(f"{zone_name} done — {len(zone_subset)} rows")

combined_with_zenith = pd.concat(zenith_dfs, ignore_index=True)

Zone_A done — 43848 rows
Zone_B done — 43848 rows
Zone_C done — 43848 rows
Zone_D done — 43848 rows


In [11]:
combined_with_zenith[["time", "zone", "solar_zenith", "solar_elevation"]].head(30)

,time,zone,solar_zenith,solar_elevation
0,2020-01-01 00:00:00+05:30,Zone_A,172.233292,-82.233292
1,2020-01-01 01:00:00+05:30,Zone_A,172.159326,-82.159326
2,2020-01-01 02:00:00+05:30,Zone_A,159.243550,-69.243550
3,2020-01-01 03:00:00+05:30,Zone_A,145.887445,-55.887445
4,2020-01-01 04:00:00+05:30,Zone_A,132.568434,-42.568434
5,2020-01-01 05:00:00+05:30,Zone_A,119.406617,-29.406617
6,2020-01-01 06:00:00+05:30,Zone_A,106.509015,-16.509015
7,2020-01-01 07:00:00+05:30,Zone_A,94.016237,-4.016237
8,2020-01-01 08:00:00+05:30,Zone_A,82.032411,7.967589
9,2020-01-01 09:00:00+05:30,Zone_A,71.154940,18.845060


In [12]:
# Save the merged dataset with zenith/elevation included
combined_with_zenith.to_csv("./data/all_zones_weather_with_zenith.csv", index=False)

print("Saved. Shape:", combined_with_zenith.shape)
print("New columns:", [c for c in combined_with_zenith.columns if c not in combined.columns])

Saved. Shape: (175392, 18)
New columns: ['solar_zenith', 'solar_elevation']


In [13]:
# Filter to daylight hours only, based on actual recorded irradiance
daylight_df = combined_with_zenith[combined_with_zenith["shortwave_radiation"] > 0].copy()

print("Original rows:", len(combined_with_zenith))
print("Daylight rows:", len(daylight_df))
print("Dropped:", len(combined_with_zenith) - len(daylight_df))
print("Daylight %:", round(len(daylight_df) / len(combined_with_zenith) * 100, 2))

daylight_df.to_csv("./data/all_zones_daylight_only.csv", index=False)
print("Saved.")

Original rows: 175392
Daylight rows: 93629
Dropped: 81763
Daylight %: 53.38
Saved.


In [14]:
daylight_df["hour"] = pd.to_datetime(daylight_df["time"]).dt.hour
print(daylight_df.groupby("hour").size())

hour
6     3675
7     7288
8     7308
9     7308
10    7308
11    7308
12    7308
13    7308
14    7308
15    7308
16    7308
17    7308
18    6287
19    3299
dtype: int64


In [15]:
# System parameters per zone
ZONE_CAPACITY_KW = 5000  # 5 MW per zone, in kW (DC nameplate capacity)

# Module parameters (standard crystalline silicon)
MODULE_EFFICIENCY = 0.19          # 19% — mid-range of 18-20%
PDC0 = ZONE_CAPACITY_KW * 1000    # DC nameplate power in watts, at STC (1000 W/m², 25°C)

# Temperature coefficient of power (standard crystalline silicon)
GAMMA_PDC = -0.004   # -0.4% per °C above 25°C (typical for c-Si)

# System losses
INVERTER_EFFICIENCY = 0.97   # 97%, mid of 96-98% range
SYSTEM_LOSSES = 0.14         # 14% — wiring, soiling baseline, mismatch, etc. (PVWatts default ballpark)

# Cell temperature model parameters (Faiman model - simple, robust, needs only ambient temp + wind)
# These are typical default coefficients used in pvlib's Faiman model
U0 = 25.0   # constant heat transfer coefficient
U1 = 6.84   # wind-dependent heat transfer coefficient

In [16]:
from pvlib import temperature

# Using Faiman model: needs poa_global (irradiance on panel) + ambient temp + wind speed
daylight_df["cell_temperature"] = temperature.faiman(
    poa_global=daylight_df["global_tilted_irradiance"],
    temp_air=daylight_df["temperature_2m"],
    wind_speed=daylight_df["wind_speed_10m"],
    u0=U0,
    u1=U1
)

In [17]:
# from pvlib import pvsystem

# daylight_df["dc_power_w"] = pvsystem.pvwatts_dc(
#     effective_irradiance=daylight_df["global_tilted_irradiance"],
#     temp_cell=daylight_df["cell_temperature"],
#     pdc0=PDC0,
#     gamma_pdc=GAMMA_PDC
# )

# # Convert to kW for readability
# daylight_df["dc_power_kw"] = daylight_df["dc_power_w"] / 1000

In [18]:
# daylight_df["ac_power_kw"] = (
#     daylight_df["dc_power_kw"] * INVERTER_EFFICIENCY * (1 - SYSTEM_LOSSES)
# )

# # Clip negative values (can occur from model edge cases at very low irradiance) to zero
# daylight_df["ac_power_kw"] = daylight_df["ac_power_kw"].clip(lower=0)

In [19]:
# daylight_df[["time", "zone", "global_tilted_irradiance", "temperature_2m", 
#              "cell_temperature", "dc_power_kw", "ac_power_kw"]].sort_values("ac_power_kw", ascending=False).head(20)

In [20]:
# # Save the dataset with plant output included
# daylight_df.to_csv("./data/all_zones_with_output.csv", index=False)

# print("Saved. Shape:", daylight_df.shape)
# print("Approx file size:", round(os.path.getsize("./data/all_zones_with_output.csv") / (1024*1024), 2), "MB")

In [21]:
# # Performance Ratio = Actual Output / Expected (theoretical) Output
# # Right now, we only have "expected output" (ac_power_kw) from the physics model.
# # Once real sensor/actual output data is available, plug it into 'actual_output_kw' below.

# # Placeholder column - to be replaced with real sensor readings when available
# daylight_df["actual_output_kw"] = None  # intentionally empty for now

# def calculate_pr(actual_kw, expected_kw):
#     """
#     Performance Ratio = Actual / Expected
#     Returns None where actual is missing (no real sensor data yet)
#     """
#     if actual_kw is None or expected_kw == 0:
#         return None
#     return actual_kw / expected_kw

# # This will produce all None for now, since actual_output_kw is empty
# daylight_df["performance_ratio"] = daylight_df.apply(
#     lambda row: calculate_pr(row["actual_output_kw"], row["ac_power_kw"]),
#     axis=1
# )

# print(daylight_df[["time", "zone", "ac_power_kw", "actual_output_kw", "performance_ratio"]].head())

In [22]:
# # Anomaly flag - will only trigger once actual_output_kw has real values
# ANOMALY_THRESHOLD = 0.85  # flag if PR drops below 85% of expected

# daylight_df["is_anomaly"] = daylight_df["performance_ratio"].apply(
#     lambda pr: pr < ANOMALY_THRESHOLD if pr is not None else None
# )

In [23]:
import numpy as np

# Get Zone_A's full 24-hour data (unfiltered, includes night)
zone_a_full = combined_with_zenith[combined_with_zenith["zone"] == "Zone_A"].copy()
zone_a_full = zone_a_full.sort_values("time").reset_index(drop=True)

# Recompute cell temperature and AC power for the FULL dataset (same formula as before)
from pvlib import temperature, pvsystem

zone_a_full["cell_temperature"] = temperature.faiman(
    poa_global=zone_a_full["global_tilted_irradiance"],
    temp_air=zone_a_full["temperature_2m"],
    wind_speed=zone_a_full["wind_speed_10m"],
    u0=U0,
    u1=U1
)

zone_a_full["dc_power_kw"] = pvsystem.pvwatts_dc(
    effective_irradiance=zone_a_full["global_tilted_irradiance"],
    temp_cell=zone_a_full["cell_temperature"],
    pdc0=PDC0,
    gamma_pdc=GAMMA_PDC
) / 1000

zone_a_full["ac_power_kw"] = (
    zone_a_full["dc_power_kw"] * INVERTER_EFFICIENCY * (1 - SYSTEM_LOSSES)
).clip(lower=0)

print("Rows:", len(zone_a_full))
print("Any NaNs in ac_power_kw?", zone_a_full["ac_power_kw"].isna().sum())

Rows: 43848
Any NaNs in ac_power_kw? 0


In [24]:
from scipy.fft import fft, fftfreq

# Extract the signal
signal = zone_a_full["ac_power_kw"].values
n = len(signal)

# Sampling: 1 sample per hour
sample_spacing_hours = 1

# Compute FFT
fft_values = fft(signal)
frequencies = fftfreq(n, d=sample_spacing_hours)  # frequencies in cycles per hour

# We only care about positive frequencies (FFT output is symmetric)
positive_mask = frequencies > 0
frequencies_positive = frequencies[positive_mask]
amplitudes = np.abs(fft_values[positive_mask])

# Convert frequency (cycles/hour) to period (hours) for easier interpretation
periods_hours = 1 / frequencies_positive

# Build a dataframe sorted by amplitude (strongest cycles first)
fft_df = pd.DataFrame({
    "period_hours": periods_hours,
    "period_days": periods_hours / 24,
    "amplitude": amplitudes
}).sort_values("amplitude", ascending=False)

fft_df.head(10)

,period_hours,period_days,amplitude
1826,24.000000,1.000000,3.194723e+07
3653,12.000000,0.500000,1.563981e+07
5480,8.000000,0.333333,2.557123e+06
1836,23.869352,0.994556,2.127243e+06
3648,12.016443,0.500685,2.122711e+06
1821,24.065862,1.002744,1.989934e+06
7307,6.000000,0.250000,1.893601e+06
4,8769.600000,365.400000,1.789049e+06
9,4384.800000,182.700000,1.773175e+06
1831,23.934498,0.997271,1.581773e+06


In [25]:
# Step 1: Calculate average power output for each hour-of-day (across all 5 years)
zone_a_full["hour"] = zone_a_full["time"].dt.hour

hourly_avg = zone_a_full.groupby("hour")["ac_power_kw"].mean()
print("Average output by hour of day:")
print(hourly_avg)

Average output by hour of day:
hour
0        0.000000
1        0.000000
2        0.000000
3        0.000000
4        0.000000
5        0.000000
6       24.140507
7      250.436492
8      904.624933
9     1698.109929
10    2392.557526
11    2875.376387
12    3081.963049
13    3016.025319
14    2704.204004
15    2189.537759
16    1505.628374
17     763.823768
18     199.565334
19      18.098688
20       0.000000
21       0.000000
22       0.000000
23       0.000000
Name: ac_power_kw, dtype: float64


In [26]:
# Step 2: Subtract the hour-of-day average from each row -> residual
zone_a_full["hourly_avg"] = zone_a_full["hour"].map(hourly_avg)
zone_a_full["residual"] = zone_a_full["ac_power_kw"] - zone_a_full["hourly_avg"]

print(zone_a_full[["time", "hour", "ac_power_kw", "hourly_avg", "residual"]].head(15))

                        time  hour  ac_power_kw   hourly_avg    residual
0  2020-01-01 00:00:00+05:30     0     0.000000     0.000000    0.000000
1  2020-01-01 01:00:00+05:30     1     0.000000     0.000000    0.000000
2  2020-01-01 02:00:00+05:30     2     0.000000     0.000000    0.000000
3  2020-01-01 03:00:00+05:30     3     0.000000     0.000000    0.000000
4  2020-01-01 04:00:00+05:30     4     0.000000     0.000000    0.000000
5  2020-01-01 05:00:00+05:30     5     0.000000     0.000000    0.000000
6  2020-01-01 06:00:00+05:30     6     0.000000    24.140507  -24.140507
7  2020-01-01 07:00:00+05:30     7     0.000000   250.436492 -250.436492
8  2020-01-01 08:00:00+05:30     8   535.574077   904.624933 -369.050856
9  2020-01-01 09:00:00+05:30     9  1485.426158  1698.109929 -212.683771
10 2020-01-01 10:00:00+05:30    10  2307.899982  2392.557526  -84.657544
11 2020-01-01 11:00:00+05:30    11  3027.551160  2875.376387  152.174774
12 2020-01-01 12:00:00+05:30    12  3370.452174  30

In [27]:
from statsmodels.tsa.arima.model import ARIMA

# Drop the all-zero night rows for ARMA - they're not real variation, just structural zeros
# (residual is always exactly 0 at night since both actual and average are 0)
residual_series = zone_a_full[zone_a_full["hour"].between(6, 19)]["residual"].reset_index(drop=True)

print("Series length:", len(residual_series))

# Fit ARMA(2,2) - using ARIMA with d=0 (no differencing, since we already deseasonalized)
model = ARIMA(residual_series, order=(2, 0, 2))
fitted_model = model.fit()

print(fitted_model.summary())

Series length: 25578
                               SARIMAX Results                                
Dep. Variable:               residual   No. Observations:                25578
Model:                 ARIMA(2, 0, 2)   Log Likelihood             -175818.327
Date:                Mon, 06 Jul 2026   AIC                         351648.654
Time:                        16:31:17   BIC                         351697.551
Sample:                             0   HQIC                        351664.463
                              - 25578                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0048     38.853     -0.000      1.000     -76.155      76.146
ar.L1          1.7021      0.005    352.159      0.000       1.693       1.712
ar.L2         -0.7031      0.00

In [28]:
# Sort Zone_A data by time, keep only the columns we need
zone_a_sorted = zone_a_full[["time", "hour", "ac_power_kw", "residual"]].sort_values("time").reset_index(drop=True)

WINDOW_SIZE_HOURS = 7 * 24   # 7 days
STEP_SIZE_HOURS = 24          # slide forward by 1 day

print("Total rows available:", len(zone_a_sorted))
print("Window size (hours):", WINDOW_SIZE_HOURS)
print("Expected number of windows:", (len(zone_a_sorted) - WINDOW_SIZE_HOURS) // STEP_SIZE_HOURS)

Total rows available: 43848
Window size (hours): 168
Expected number of windows: 1820


In [29]:
from scipy.fft import fft, fftfreq
import numpy as np

window_summaries = []

for start_idx in range(0, len(zone_a_sorted) - WINDOW_SIZE_HOURS, STEP_SIZE_HOURS):
    
    window = zone_a_sorted.iloc[start_idx : start_idx + WINDOW_SIZE_HOURS]
    
    window_start = window["time"].iloc[0]
    window_end = window["time"].iloc[-1]
    signal = window["ac_power_kw"].values
    residual = window["residual"].values
    
    # --- FFT on AC power output ---
    n = len(signal)
    fft_vals = np.abs(fft(signal))
    freqs = fftfreq(n, d=1)  # cycles per hour
    
    # Only positive frequencies
    pos_mask = freqs > 0
    pos_freqs = freqs[pos_mask]
    pos_amps = fft_vals[pos_mask]
    
    # Find dominant frequency and convert to period
    dominant_freq = pos_freqs[np.argmax(pos_amps)]
    dominant_period_hours = 1 / dominant_freq
    dominant_amplitude = pos_amps.max()
    
    # --- Residual statistics ---
    residual_mean = residual.mean()
    residual_std = residual.std()
    residual_max_dev = residual.max()
    residual_min_dev = residual.min()
    abs_max_dev = np.abs(residual).max()
    
    # --- Deviation from historical norm ---
    # Compare this window's mean daytime output to overall Zone_A daytime average
    daytime_window = window[window["hour"].between(8, 17)]
    daytime_avg_this_window = daytime_window["ac_power_kw"].mean()
    
    window_summaries.append({
        "window_start": window_start,
        "window_end": window_end,
        "dominant_period_hours": round(dominant_period_hours, 2),
        "dominant_amplitude": round(dominant_amplitude, 2),
        "residual_mean": round(residual_mean, 2),
        "residual_std": round(residual_std, 2),
        "residual_max_dev": round(residual_max_dev, 2),
        "residual_min_dev": round(residual_min_dev, 2),
        "abs_max_deviation_kw": round(abs_max_dev, 2),
        "daytime_avg_kw": round(daytime_avg_this_window, 2),
    })

summaries_df = pd.DataFrame(window_summaries)
print(f"Generated {len(summaries_df)} window summaries")
summaries_df.head(5)

Generated 1820 window summaries


,window_start,window_end,dominant_period_hours,dominant_amplitude,residual_mean,residual_std,residual_max_dev,residual_min_dev,abs_max_deviation_kw,daytime_avg_kw
0,2020-01-01 00:00:00+05:30,2020-01-07 23:00:00+05:30,24.0,113427.54,-103.81,196.12,342.17,-807.65,807.65,1912.91
1,2020-01-02 00:00:00+05:30,2020-01-08 23:00:00+05:30,24.0,111572.26,-115.71,187.90,247.19,-807.65,807.65,1884.17
2,2020-01-03 00:00:00+05:30,2020-01-09 23:00:00+05:30,24.0,115663.58,-85.62,204.09,506.16,-807.65,807.65,1956.19
3,2020-01-04 00:00:00+05:30,2020-01-10 23:00:00+05:30,24.0,118670.78,-62.90,219.03,506.16,-807.65,807.65,2010.54
4,2020-01-05 00:00:00+05:30,2020-01-11 23:00:00+05:30,24.0,121551.90,-45.07,221.81,506.16,-807.65,807.65,2053.15


In [30]:
summaries_df.to_csv("./data/zone_a_window_summaries.csv", index=False)
print("Saved. Shape:", summaries_df.shape)

Saved. Shape: (1820, 10)


In [31]:
import ollama

# Simple test — make sure Phi-4-Mini responds correctly
response = ollama.chat(
    model="phi4-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a solar plant data summarizer. Summarize sensor data concisely using only the information provided. Always preserve exact numbers. Never add external information."
        },
        {
            "role": "user",
            "content": "Summarize this data: Irradiance was stable at 650 W/m² from 10am to 12pm. Output held steady at 2800 kW. Temperature rose from 30°C to 34°C."
        },
        {
            "role": "assistant",
            "content": "10:00-12:00: Irradiance stable at 650 W/m², output steady at 2800 kW. Temperature rose moderately from 30°C to 34°C. No anomalies observed."
        },
        {
            "role": "user",
            "content": "Summarize this data: Solar irradiance peaked at 820 W/m² at noon, then dropped sharply to 340 W/m² by 2pm due to increasing cloud cover. Plant output fell from 3800 kW to 1600 kW over the same period. Temperature was stable at 35°C throughout."
        }
    ]
)

print(response["message"]["content"])

12:00-14:00: Peak irradiance reached 820 W/m², dropped sharply to a midday low of 340 W/m² by clouds affecting plant performance (output fell from peak 3800 kW). Output reduced significantly in tandem with falling solar intensity (-2600 kW), while temperature remained constant at an average level.


In [34]:
def get_local_summary(data_text):
    """
    Send raw sensor data text to Phi-4-Mini for summarization.
    Post-process to trim verbosity before passing to large LLM.
    """
    response = ollama.chat(
        model="phi4-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a solar plant data summarizer. Summarize sensor data concisely using only the information provided. Always preserve exact numbers. Never add external information."
            },
            {
                "role": "user",
                "content": "Summarize this data: Irradiance was stable at 650 W/m² from 10am to 12pm. Output held steady at 2800 kW. Temperature rose from 30°C to 34°C."
            },
            {
                "role": "assistant",
                "content": "10:00-12:00: Irradiance stable at 650 W/m², output steady at 2800 kW. Temperature rose moderately from 30°C to 34°C. No anomalies observed."
            },
            {
                "role": "user",
                "content": f"Summarize this data: {data_text}"
            }
        ]
    )
    
    raw = response["message"]["content"]
    
    # Post-processing: keep only first 3 sentences
    sentences = raw.replace("\n", " ").split(".")
    trimmed = ". ".join(s.strip() for s in sentences[:3] if s.strip()) + "."
    
    return trimmed


# Test it on a real window from our data
test_window = daylight_df[daylight_df["zone"] == "Zone_A"].head(10)

data_text = " | ".join([
    f"{row['time'].strftime('%H:%M')}: GTI={row['global_tilted_irradiance']:.0f} W/m², "
    f"Output={row['ac_power_kw']:.0f} kW, "
    f"Temp={row['temperature_2m']:.1f}°C, "
    f"Cloud={row['cloud_cover']}%"
    for _, row in test_window.iterrows()
])

summary = get_local_summary(data_text)
print("Raw data fed in:")
print(data_text)
print()
print("Phi-4-Mini summary:")
print(summary)

AttributeError: 'str' object has no attribute 'strftime'

In [35]:
from edge_llm import format_window_for_summary, get_local_summary

# Test on same Zone_A data
test_window = daylight_df[daylight_df["zone"] == "Zone_A"].head(10)
structured_text = format_window_for_summary(test_window)
print(structured_text)

Morning (06:00-11:00): GTI rose from 121 to 712 W/m², output from 536 to 3028 kW, avg cloud cover 21%, temp 6.8 to 15.0°C. Peak: GTI 799 W/m², output 3372 kW at 12:00. Afternoon (12:00-17:00): GTI declined from 799 to 357 W/m², output from 3372 to 1519 kW, avg cloud cover 1%, temp 16.5 to 17.4°C.


In [ ]:
from edge_llm import format_window_for_summary, get_local_summary

# Test on Zone_A data
test_window = daylight_df[daylight_df["zone"] == "Zone_A"].head(10)

structured_text = format_window_for_summary(test_window)
summary = get_local_summary(structured_text)

print("=== Edge Summarization Pipeline Test ===")
print("Input rows:", len(test_window))
print()
print("Step 1 - Structured text:")
print(structured_text)
print()
print("Step 2 - Phi-4-Mini summary:")
print(summary)

=== Edge Summarization Pipeline Test ===
Input rows: 10

Step 1 - Structured text:
Morning (06:00-11:00): GTI rose from 121 to 712 W/m², output from 536 to 3028 kW, avg cloud cover 21%, temp 6.8 to 15.0°C. Peak: GTI 799 W/m², output 3372 kW at 12:00. Afternoon (12:00-17:00): GTI declined from 799 to 357 W/m², output from 3372 to 1519 kW, avg cloud cover 1%, temp 16.5 to 17.4°C.

Step 2 - Phi-4-Mini summary:
Morning Summary: 06:00–11:00 - Solar Irradiance (GTI) increased gradually reaching a peak of 712 W/m² with the highest output at noon being approximately 3028 kWh from an average cloud cover and temperature ranging between 6. 8 to 15. 0°C.


In [ ]:
daylight_df = pd.read_csv("./data/all_zones_with_output.csv")
print(daylight_df.columns.tolist())

['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_radiation', 'global_tilted_irradiance', 'sunshine_duration', 'zone', 'latitude', 'longitude', 'solar_zenith', 'solar_elevation', 'hour', 'cell_temperature', 'dc_power_w', 'dc_power_kw', 'ac_power_kw']


In [36]:
from groq import Groq
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

GROQ_API_KEY = os.getenv("API_KEY")

# Confirm it loaded (never print the actual key)
print("API key loaded:", "Yes" if GROQ_API_KEY else "No — check your .env file")

client = Groq(api_key=GROQ_API_KEY)

# Quick test
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "You are a solar plant operations assistant. Given this summary: '06:00-12:00: GTI peaked at 799 W/m², output 3372 kW at noon, cloud cover low at 1%. Morning rise steady with 21% avg cloud cover.' What is your assessment of plant performance?"
        }
    ],
    max_tokens=300
)

print(response.choices[0].message.content)

API key loaded: No — check your .env file
Based on the provided summary, my assessment of the plant performance from 06:00 to 12:00 is as follows:

1. **Irradiance Levels**: The Global Tilted Irradiance (GTI) peaked at 799 W/m², which is a relatively high value, indicating favorable sunlight conditions for solar energy production. This peak suggests that the solar panels were able to capitalize on the available sunlight during this period.

2. **Power Output**: The plant achieved a maximum output of 3372 kW at noon. This is a significant production level, suggesting that the solar panels and the overall system are functioning efficiently. However, to fully assess the performance, it would be beneficial to know the plant's capacity and compare the actual output to the potential output based on the irradiance levels.

3. **Cloud Cover**: The cloud cover was low at 1% during the peak production time, which is ideal for solar energy generation. The morning rise had an average cloud cover o

In [37]:
def build_llmsense_prompt(local_summary, window_stats, zone_name):
    """
    Build the full LLMSense prompt following:
    Prompt = Objective + Context + Data + Format
    """
    
    # ── OBJECTIVE ──────────────────────────────────────────────
    objective = """You are a Solar Plant Operations Copilot.
Your job is to assess current plant performance, identify anomalies, 
and recommend specific operator actions based on sensor data.
Be precise, factual, and actionable. Only use information provided."""

    # ── CONTEXT ────────────────────────────────────────────────
    context = f"""
Plant Information:
- Location: Jaipur, Rajasthan, India (Latitude 26.9°N)
- Zone: {zone_name}
- Nameplate Capacity: 5000 kW (5 MW) per zone
- Panel Type: Fixed-tilt crystalline silicon, tilt angle 24°, south-facing
- Grid Contract Minimum: 1500 kW during daylight hours
- Battery Reserve Threshold: Activate if output expected below 1500 kW for >2 hours
- Normal Performance Ratio: 0.75 - 0.85 (actual/theoretical output)
- Season context: Jaipur receives peak irradiance in March-April, 
  lowest in December-January, monsoon cloud disruption June-September
"""

    # ── DATA ───────────────────────────────────────────────────
    data = f"""
Current Window Data:
Period: {window_stats['window_start']} to {window_stats['window_end']}

Edge LLM Summary (Phi-4-Mini):
{local_summary}

Statistical Summary (7-day sliding window):
- Dominant cycle period: {window_stats['dominant_period_hours']} hours
- Daytime average output: {window_stats['daytime_avg_kw']} kW
- Residual std deviation: {window_stats['residual_std']} kW
- Maximum deviation from typical: {window_stats['abs_max_deviation_kw']} kW
- Residual mean (above/below seasonal norm): {window_stats['residual_mean']} kW
"""

    # ── FORMAT ─────────────────────────────────────────────────
    output_format = """
Respond in exactly this structure:

STATUS: [Normal / Warning / Critical]
PERFORMANCE: [Current output vs expected, as a percentage]
ANOMALY: [None detected / describe anomaly with exact numbers]
RECOMMENDATION: [Specific action for operator, or "No action required"]
URGENCY: [Immediate / Within 1 hour / Monitor / None]

NARRATIVE:
[One paragraph: what is happening, why, what the operator should know, 
what will likely happen next based on current trends. 
Use exact numbers. Maximum 5 sentences.]
"""

    # ── ASSEMBLE ───────────────────────────────────────────────
    full_prompt = f"{objective}\n{context}\n{data}\n{output_format}"
    return full_prompt


# ── TEST: Run on our real Zone_A window ────────────────────────
test_stats = summaries_df.iloc[0].to_dict()

full_prompt = build_llmsense_prompt(
    local_summary=summary,
    window_stats=test_stats,
    zone_name="Zone_A"
)

print("=== FULL PROMPT ===")
print(full_prompt)

NameError: name 'summary' is not defined

In [38]:
def get_cloud_reasoning(prompt, max_tokens=600):
    """
    Send the full LLMSense prompt to Llama 3.3 70B via Groq
    for high-level reasoning and operator recommendations.
    """
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=max_tokens,
        temperature=0.3  # lower temperature = more consistent, factual output
    )
    return response.choices[0].message.content


# Run the full pipeline end to end
print("=== LLMSENSE PIPELINE OUTPUT ===")
print(f"Zone: Zone_A")
print(f"Window: {test_stats['window_start']} to {test_stats['window_end']}")
print()

reasoning = get_cloud_reasoning(full_prompt)
print(reasoning)

=== LLMSENSE PIPELINE OUTPUT ===
Zone: Zone_A
Window: 2020-01-01 00:00:00+05:30 to 2020-01-07 23:00:00+05:30



NameError: name 'full_prompt' is not defined

In [39]:
def run_pipeline(zone_name, window_index):
    """
    Full LLMSense pipeline for a given zone and window index.
    
    Steps:
    1. Get window statistics from pre-computed summaries
    2. Get raw daylight data for that window
    3. Pre-aggregate into structured text
    4. Get Phi-4-Mini edge summary
    5. Build LLMSense prompt
    6. Get Llama 3.3 70B reasoning
    7. Return full output
    """
    
    # ── Step 1: Get window stats ───────────────────────────────
    window_stats = summaries_df.iloc[window_index].to_dict()
    window_start = pd.to_datetime(window_stats["window_start"])
    window_end = pd.to_datetime(window_stats["window_end"])
    
    # ── Step 2: Get raw data for this window ───────────────────
    zone_data = daylight_df[daylight_df["zone"] == zone_name].copy()
    zone_data["time"] = pd.to_datetime(zone_data["time"])
    
    window_data = zone_data[
        (zone_data["time"] >= window_start) &
        (zone_data["time"] <= window_end)
    ]
    
    if len(window_data) == 0:
        print(f"No data found for {zone_name}, window {window_index}")
        return None
    
    # ── Step 3: Pre-aggregate ──────────────────────────────────
    structured_text = format_window_for_summary(window_data)
    
    # ── Step 4: Phi-4-Mini edge summary ───────────────────────
    local_summary = get_local_summary(structured_text)
    
    # ── Step 5: Build LLMSense prompt ─────────────────────────
    prompt = build_llmsense_prompt(local_summary, window_stats, zone_name)
    
    # ── Step 6: Llama 3.3 70B reasoning ───────────────────────
    reasoning = get_cloud_reasoning(prompt)
    
    # ── Step 7: Return structured output ──────────────────────
    result = {
        "zone": zone_name,
        "window_start": window_stats["window_start"],
        "window_end": window_stats["window_end"],
        "structured_text": structured_text,
        "local_summary": local_summary,
        "reasoning": reasoning
    }
    
    return result


# ── Test on a few different windows ───────────────────────────
def print_pipeline_output(result):
    print("=" * 60)
    print(f"Zone: {result['zone']}")
    print(f"Window: {result['window_start']} → {result['window_end']}")
    print()
    print("── Edge Summary (Phi-4-Mini) ──")
    print(result["local_summary"])
    print()
    print("── Cloud Reasoning (Llama 3.3 70B) ──")
    print(result["reasoning"])
    print("=" * 60)


# # Test on 3 different windows to validate consistency
# # Window 0  — January 2020 (winter)
# # Window 90 — April 2020 (peak season)
# # Window 180 — July 2020 (monsoon)

# for idx in [0, 90, 180]:
#     result = run_pipeline("Zone_A", idx)
#     if result:
#         print_pipeline_output(result)
#     print()

In [40]:
## RAG Layer — LangChain Setup

In [41]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

load_dotenv()

# ── LLMs ──────────────────────────────────────────────────────

# Edge LLM (Phi-4-Mini via Ollama) — stays as is, Ollama not in LangChain scope
# We keep the raw ollama client for this

# Cloud LLM — refactored to LangChain ChatGroq
llm = ChatGroq(
    api_key=os.getenv("API_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0.3
)

# ── Embeddings ─────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("LangChain setup complete")
print("LLM:", llm.model_name)
print("Embeddings:", embeddings.model_name)

c:\Users\divya\Projects\NexTurn Prroject\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\divya\AppData\Local\Temp\ipykernel_54672\1095493646.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5359.50it/s]


LangChain setup complete
LLM: llama-3.3-70b-versatile
Embeddings: sentence-transformers/all-MiniLM-L6-v2


In [42]:

# ── 1. Window Summary Documents ────────────────────────────────
def window_row_to_document(row):
    """Convert a window summary row into a LangChain Document."""
    
    window_start = pd.to_datetime(row["window_start"])
    month = window_start.month
    season = (
        "Winter" if month in [11, 12, 1, 2] else
        "Spring/Peak" if month in [3, 4, 5] else
        "Monsoon" if month in [6, 7, 8, 9] else
        "Autumn"
    )
    
    content = (
        f"Window: {row['window_start']} to {row['window_end']}. "
        f"Season: {season}. "
        f"Dominant cycle: {row['dominant_period_hours']} hours. "
        f"Daytime average output: {row['daytime_avg_kw']} kW. "
        f"Residual std deviation: {row['residual_std']} kW. "
        f"Max deviation from typical: {row['abs_max_deviation_kw']} kW. "
        f"Residual mean vs seasonal norm: {row['residual_mean']} kW."
    )
    
    metadata = {
        "type": "window_summary",
        "window_start": str(row["window_start"]),
        "window_end": str(row["window_end"]),
        "month": month,
        "season": season,
        "daytime_avg_kw": float(row["daytime_avg_kw"]),
        "residual_std": float(row["residual_std"]),
        "abs_max_deviation_kw": float(row["abs_max_deviation_kw"])
    }
    
    return Document(page_content=content, metadata=metadata)


window_docs = [window_row_to_document(row) for _, row in summaries_df.iterrows()]

print(f"Window summary documents: {len(window_docs)}")
print()
print("Sample document:")
print(window_docs[0].page_content)
print()
print("Sample metadata:")
print(window_docs[0].metadata)

Window summary documents: 1820

Sample document:
Window: 2020-01-01 00:00:00+05:30 to 2020-01-07 23:00:00+05:30. Season: Winter. Dominant cycle: 24.0 hours. Daytime average output: 1912.91 kW. Residual std deviation: 196.12 kW. Max deviation from typical: 807.65 kW. Residual mean vs seasonal norm: -103.81 kW.

Sample metadata:
{'type': 'window_summary', 'window_start': '2020-01-01 00:00:00+05:30', 'window_end': '2020-01-07 23:00:00+05:30', 'month': 1, 'season': 'Winter', 'daytime_avg_kw': 1912.91, 'residual_std': 196.12, 'abs_max_deviation_kw': 807.65}


In [43]:
# ── 2. Seasonal Baseline Documents ────────────────────────────
# One document per month summarizing typical performance
# across all 5 years for that month

seasonal_docs = []

month_names = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December"
}

season_map = {
    1: "Winter", 2: "Winter", 3: "Spring/Peak", 4: "Spring/Peak",
    5: "Spring/Peak", 6: "Monsoon", 7: "Monsoon", 8: "Monsoon",
    9: "Monsoon", 10: "Autumn", 11: "Winter", 12: "Winter"
}

# Add month column to summaries_df temporarily
summaries_df["month"] = pd.to_datetime(summaries_df["window_start"]).dt.month

for month_num in range(1, 13):
    month_data = summaries_df[summaries_df["month"] == month_num]
    
    if len(month_data) == 0:
        continue
    
    avg_output = month_data["daytime_avg_kw"].mean()
    std_output = month_data["residual_std"].mean()
    avg_max_dev = month_data["abs_max_deviation_kw"].mean()
    min_output = month_data["daytime_avg_kw"].min()
    max_output = month_data["daytime_avg_kw"].max()
    
    content = (
        f"Seasonal Baseline for {month_names[month_num]} "
        f"({season_map[month_num]} season). "
        f"Based on 5 years of historical data (2020-2024). "
        f"Typical daytime average output: {avg_output:.1f} kW. "
        f"Output range: {min_output:.1f} to {max_output:.1f} kW. "
        f"Typical residual std deviation: {std_output:.1f} kW. "
        f"Typical max deviation from norm: {avg_max_dev:.1f} kW. "
        f"Plant nameplate capacity: 5000 kW per zone. "
        f"Grid contract minimum: 1500 kW during daylight hours."
    )
    
    metadata = {
        "type": "seasonal_baseline",
        "month": month_num,
        "month_name": month_names[month_num],
        "season": season_map[month_num],
        "avg_daytime_output_kw": round(avg_output, 1),
        "min_output_kw": round(min_output, 1),
        "max_output_kw": round(max_output, 1)
    }
    
    seasonal_docs.append(Document(page_content=content, metadata=metadata))

print(f"Seasonal baseline documents: {len(seasonal_docs)}")
print()
for doc in seasonal_docs:
    print(doc.page_content)
    print()
    

Seasonal baseline documents: 12

Seasonal Baseline for January (Winter season). Based on 5 years of historical data (2020-2024). Typical daytime average output: 2130.5 kW. Output range: 1332.8 to 2486.4 kW. Typical residual std deviation: 298.5 kW. Typical max deviation from norm: 1224.5 kW. Plant nameplate capacity: 5000 kW per zone. Grid contract minimum: 1500 kW during daylight hours.

Seasonal Baseline for February (Winter season). Based on 5 years of historical data (2020-2024). Typical daytime average output: 2433.4 kW. Output range: 2028.5 to 2632.1 kW. Typical residual std deviation: 286.1 kW. Typical max deviation from norm: 974.3 kW. Plant nameplate capacity: 5000 kW per zone. Grid contract minimum: 1500 kW during daylight hours.

Seasonal Baseline for March (Spring/Peak season). Based on 5 years of historical data (2020-2024). Typical daytime average output: 2433.8 kW. Output range: 1801.1 to 2732.7 kW. Typical residual std deviation: 289.2 kW. Typical max deviation from nor

In [44]:
# ── Combine all documents ──────────────────────────────────────
all_docs = window_docs + seasonal_docs
print(f"Total documents to index: {len(all_docs)}")

# ── Build FAISS vector store ───────────────────────────────────
print("Building FAISS index...")
vectorstore = FAISS.from_documents(all_docs, embeddings)
print("FAISS index built.")

# ── BM25 Retriever ─────────────────────────────────────────────
bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 3

# ── FAISS Retriever ────────────────────────────────────────────
faiss_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# ── Hybrid Retriever (same weights as your existing RAG) ───────
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]
)

print("Hybrid retriever ready.")

# ── Quick test retrieval ───────────────────────────────────────
test_query = "monsoon season July low output cloud cover"
results = hybrid_retriever.invoke(test_query)

print(f"\nTest query: '{test_query}'")
print(f"Retrieved {len(results)} documents:")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] Type: {doc.metadata['type']} | Season: {doc.metadata.get('season', 'N/A')}")
    print(f"     {doc.page_content[:150]}...")

Total documents to index: 1832
Building FAISS index...
FAISS index built.
Hybrid retriever ready.

Test query: 'monsoon season July low output cloud cover'
Retrieved 5 documents:

[1] Type: seasonal_baseline | Season: Monsoon
     Seasonal Baseline for July (Monsoon season). Based on 5 years of historical data (2020-2024). Typical daytime average output: 1797.5 kW. Output range:...

[2] Type: window_summary | Season: Monsoon
     Window: 2021-08-18 00:00:00+05:30 to 2021-08-24 23:00:00+05:30. Season: Monsoon. Dominant cycle: 24.0 hours. Daytime average output: 1937.61 kW. Resid...

[3] Type: window_summary | Season: Monsoon
     Window: 2021-08-17 00:00:00+05:30 to 2021-08-23 23:00:00+05:30. Season: Monsoon. Dominant cycle: 24.0 hours. Daytime average output: 1958.29 kW. Resid...

[4] Type: window_summary | Season: Winter
     Window: 2024-12-13 00:00:00+05:30 to 2024-12-19 23:00:00+05:30. Season: Winter. Dominant cycle: 24.0 hours. Daytime average output: 2195.7 kW. Residua...

[5] Ty

In [45]:
# Save FAISS index to disk
vectorstore.save_local("./data/faiss_index")
print("FAISS index saved to ./data/faiss_index")

# Test loading it back
vectorstore_loaded = FAISS.load_local(
    "./data/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print("FAISS index loaded successfully")
print(f"Index contains: {vectorstore_loaded.index.ntotal} vectors")

FAISS index saved to ./data/faiss_index
FAISS index loaded successfully
Index contains: 1832 vectors


In [46]:
def get_rag_context(query, retriever, top_k=3):
    """
    Retrieve relevant historical context for a given query.
    Returns formatted string ready to inject into LLMSense prompt.
    """
    docs = retriever.invoke(query)
    
    # Filter to top_k most relevant
    docs = docs[:top_k]
    
    formatted = []
    for i, doc in enumerate(docs):
        doc_type = doc.metadata.get("type", "unknown")
        season = doc.metadata.get("season", "unknown")
        formatted.append(
            f"[{i+1}] ({doc_type} | {season})\n{doc.page_content}"
        )
    
    return "\n\n".join(formatted)


def build_llmsense_prompt_with_rag(local_summary, window_stats, zone_name, rag_context):
    """
    Extended LLMSense prompt — now includes RAG historical context.
    Prompt = Objective + Context + Data + Historical Context + Format
    """

    # ── OBJECTIVE ──────────────────────────────────────────────
    objective = """You are a Solar Plant Operations Copilot.
Your job is to assess current plant performance, identify anomalies,
and recommend specific operator actions based on sensor data.
Be precise, factual, and actionable. Only use information provided."""

    # ── CONTEXT ────────────────────────────────────────────────
    context = f"""
Plant Information:
- Location: Jaipur, Rajasthan, India (Latitude 26.9°N)
- Zone: {zone_name}
- Nameplate Capacity: 5000 kW (5 MW) per zone
- Panel Type: Fixed-tilt crystalline silicon, tilt angle 24°, south-facing
- Grid Contract Minimum: 1500 kW during daylight hours
- Battery Reserve Threshold: Activate if output expected below 1500 kW for >2 hours
- Normal Performance Ratio: 0.75 - 0.85
- Season context: Jaipur receives peak irradiance in March-April,
  lowest in December-January, monsoon cloud disruption June-September
"""

    # ── CURRENT DATA ───────────────────────────────────────────
    data = f"""
Current Window Data:
Period: {window_stats['window_start']} to {window_stats['window_end']}

Edge LLM Summary (Phi-4-Mini):
{local_summary}

Statistical Summary (7-day sliding window):
- Dominant cycle period: {window_stats['dominant_period_hours']} hours
- Daytime average output: {window_stats['daytime_avg_kw']} kW
- Residual std deviation: {window_stats['residual_std']} kW
- Maximum deviation from typical: {window_stats['abs_max_deviation_kw']} kW
- Residual mean (above/below seasonal norm): {window_stats['residual_mean']} kW
"""

    # ── HISTORICAL CONTEXT (RAG) ────────────────────────────────
    historical = f"""
Historical Context (retrieved from 5-year pattern database):
{rag_context}
"""

    # ── FORMAT ─────────────────────────────────────────────────
    output_format = """
Respond in exactly this structure:

STATUS: [Normal / Warning / Critical]
PERFORMANCE: [Current output vs expected, as a percentage]
ANOMALY: [None detected / describe anomaly with exact numbers]
HISTORICAL MATCH: [Most similar historical pattern and what happened next]
RECOMMENDATION: [Specific action for operator, or "No action required"]
URGENCY: [Immediate / Within 1 hour / Monitor / None]

NARRATIVE:
[One paragraph: what is happening, why, what happened in similar past situations,
what the operator should do now. Use exact numbers. Maximum 5 sentences.]
"""

    full_prompt = f"{objective}\n{context}\n{data}\n{historical}\n{output_format}"
    return full_prompt


def run_pipeline_with_rag(zone_name, window_index):
    """
    Full LLMSense + RAG pipeline for a given zone and window index.
    """

    # Step 1: Get window stats
    window_stats = summaries_df.iloc[window_index].to_dict()
    window_start = pd.to_datetime(window_stats["window_start"])
    window_end = pd.to_datetime(window_stats["window_end"])

    # Step 2: Get raw data for this window
    zone_data = daylight_df[daylight_df["zone"] == zone_name].copy()
    zone_data["time"] = pd.to_datetime(zone_data["time"])
    window_data = zone_data[
        (zone_data["time"] >= window_start) &
        (zone_data["time"] <= window_end)
    ]

    if len(window_data) == 0:
        print(f"No data found for {zone_name}, window {window_index}")
        return None

    # Step 3: Pre-aggregate
    structured_text = format_window_for_summary(window_data)

    # Step 4: Phi-4-Mini edge summary
    local_summary = get_local_summary(structured_text)

    # Step 5: RAG retrieval
    # Build query from current window context for relevant retrieval
    month = window_start.month
    season = window_stats.get("season", "unknown")
    rag_query = (
        f"{season} season month {month} "
        f"daytime output {window_stats['daytime_avg_kw']:.0f} kW "
        f"deviation {window_stats['residual_mean']:.0f} kW"
    )
    rag_context = get_rag_context(rag_query, hybrid_retriever, top_k=3)

    # Step 6: Build RAG-augmented LLMSense prompt
    prompt = build_llmsense_prompt_with_rag(
        local_summary, window_stats, zone_name, rag_context
    )

    # Step 7: Llama 3.3 70B reasoning
    reasoning = get_cloud_reasoning(prompt)

    result = {
        "zone": zone_name,
        "window_start": window_stats["window_start"],
        "window_end": window_stats["window_end"],
        "structured_text": structured_text,
        "local_summary": local_summary,
        "rag_query": rag_query,
        "rag_context": rag_context,
        "reasoning": reasoning
    }

    return result


# ── Test on same 3 windows as before ──────────────────────────
for idx in [0, 90, 180]:
    result = run_pipeline_with_rag("Zone_A", idx)
    if result:
        print("=" * 60)
        print(f"Zone: {result['zone']}")
        print(f"Window: {result['window_start']} → {result['window_end']}")
        print()
        print("── RAG Query ──")
        print(result["rag_query"])
        print()
        print("── Cloud Reasoning (Llama 3.3 70B + RAG) ──")
        print(result["reasoning"])
        print("=" * 60)
        print()

Zone: Zone_A
Window: 2020-01-01 00:00:00+05:30 → 2020-01-07 23:00:00+05:30

── RAG Query ──
unknown season month 1 daytime output 1913 kW deviation -104 kW

── Cloud Reasoning (Llama 3.3 70B + RAG) ──
STATUS: Warning
PERFORMANCE: 127% (1912.91 kW vs expected 1500 kW)
ANOMALY: Daytime average output is 103.81 kW below seasonal norm
HISTORICAL MATCH: Most similar historical pattern is the Spring/Peak season, where daytime average output was around 2202.08 kW, and the plant performed within expected norms after a brief period of cloud disruption
RECOMMENDATION: Monitor cloud cover and be prepared to activate battery reserve if output is expected to fall below 1500 kW for more than 2 hours
URGENCY: Monitor

NARRATIVE: The current output of 1912.91 kW is above the grid contract minimum of 1500 kW, but the daytime average output is 103.81 kW below the seasonal norm, indicating a potential issue. The historical data from the Spring/Peak season shows that the plant can recover from brief perio

In [47]:
# Fix 1 — Add season to summaries_df
summaries_df["season"] = summaries_df["month"].map({
    1: "Winter", 2: "Winter", 3: "Spring/Peak", 4: "Spring/Peak",
    5: "Spring/Peak", 6: "Monsoon", 7: "Monsoon", 8: "Monsoon",
    9: "Monsoon", 10: "Autumn", 11: "Winter", 12: "Winter"
})

# Fix 2 — Build seasonal expected output lookup
seasonal_lookup = {
    doc.metadata["month"]: doc.metadata["avg_daytime_output_kw"]
    for doc in seasonal_docs
}

print("Seasonal expected output lookup:")
for month, output in sorted(seasonal_lookup.items()):
    print(f"  Month {month:2d}: {output} kW")

Seasonal expected output lookup:
  Month  1: 2130.5 kW
  Month  2: 2433.4 kW
  Month  3: 2433.8 kW
  Month  4: 2347.2 kW
  Month  5: 2081.7 kW
  Month  6: 1895.8 kW
  Month  7: 1797.5 kW
  Month  8: 1883.3 kW
  Month  9: 2067.7 kW
  Month 10: 2235.1 kW
  Month 11: 2061.1 kW
  Month 12: 2022.5 kW


In [48]:
def build_llmsense_prompt_with_rag(local_summary, window_stats, zone_name, rag_context, expected_output_kw):
    """
    Updated LLMSense prompt — now includes explicit seasonal expected output.
    Prompt = Objective + Context + Data + Historical Context + Format
    """

    # Calculate performance ratio explicitly so LLM doesn't guess
    actual_output = window_stats["daytime_avg_kw"]
    performance_pct = round((actual_output / expected_output_kw) * 100, 1)

    # ── OBJECTIVE ──────────────────────────────────────────────
    objective = """You are a Solar Plant Operations Copilot.
Your job is to assess current plant performance, identify anomalies,
and recommend specific operator actions based on sensor data.
Be precise, factual, and actionable. Only use information provided."""

    # ── CONTEXT ────────────────────────────────────────────────
    context = f"""
Plant Information:
- Location: Jaipur, Rajasthan, India (Latitude 26.9°N)
- Zone: {zone_name}
- Nameplate Capacity: 5000 kW (5 MW) per zone
- Panel Type: Fixed-tilt crystalline silicon, tilt angle 24°, south-facing
- Grid Contract Minimum: 1500 kW during daylight hours
- Battery Reserve Threshold: Activate if output expected below 1500 kW for >2 hours
- Normal Performance Ratio: 0.75 - 0.85
- Season context: Jaipur receives peak irradiance in March-April,
  lowest in December-January, monsoon cloud disruption June-September
"""

    # ── CURRENT DATA ───────────────────────────────────────────
    data = f"""
Current Window Data:
Period: {window_stats['window_start']} to {window_stats['window_end']}
Season: {window_stats.get('season', 'Unknown')}

Edge LLM Summary (Phi-4-Mini):
{local_summary}

Statistical Summary (7-day sliding window):
- Dominant cycle period: {window_stats['dominant_period_hours']} hours
- Actual daytime average output: {actual_output} kW
- Seasonal expected output (historical 5-year average for this month): {expected_output_kw} kW
- Performance vs seasonal expectation: {performance_pct}%
- Residual std deviation: {window_stats['residual_std']} kW
- Maximum single-hour deviation: {window_stats['abs_max_deviation_kw']} kW
- Residual mean vs seasonal norm: {window_stats['residual_mean']} kW

Performance Assessment Guide:
- Above 95%: Normal — plant performing well
- 85-95%: Monitor — slight underperformance
- 75-85%: Warning — investigate cause
- Below 75%: Critical — immediate action required
"""

    # ── HISTORICAL CONTEXT (RAG) ────────────────────────────────
    historical = f"""
Historical Context (retrieved from 5-year pattern database):
{rag_context}
"""

    # ── FORMAT ─────────────────────────────────────────────────
    output_format = """
Respond in exactly this structure:

STATUS: [Normal / Warning / Critical] — based on performance % guide above
PERFORMANCE: [actual output] kW vs [expected output] kW = [performance %]%
ANOMALY: [None detected / describe anomaly with exact numbers]
HISTORICAL MATCH: [Most similar historical pattern and what happened next]
RECOMMENDATION: [Specific action for operator, or "No action required"]
URGENCY: [Immediate / Within 1 hour / Monitor / None]

NARRATIVE:
[One paragraph: what is happening, why, what happened in similar past situations,
what the operator should do now. Use exact numbers. Maximum 5 sentences.]
"""

    full_prompt = f"{objective}\n{context}\n{data}\n{historical}\n{output_format}"
    return full_prompt


def run_pipeline_with_rag(zone_name, window_index):
    """
    Full LLMSense + RAG pipeline — with explicit seasonal expected output.
    """
    # Step 1: Get window stats
    window_stats = summaries_df.iloc[window_index].to_dict()
    window_start = pd.to_datetime(window_stats["window_start"])
    window_end = pd.to_datetime(window_stats["window_end"])
    month = int(window_start.month)

    # Step 2: Get raw data for this window
    zone_data = daylight_df[daylight_df["zone"] == zone_name].copy()
    zone_data["time"] = pd.to_datetime(zone_data["time"])
    window_data = zone_data[
        (zone_data["time"] >= window_start) &
        (zone_data["time"] <= window_end)
    ]

    if len(window_data) == 0:
        print(f"No data found for {zone_name}, window {window_index}")
        return None

    # Step 3: Pre-aggregate
    structured_text = format_window_for_summary(window_data)

    # Step 4: Phi-4-Mini edge summary
    local_summary = get_local_summary(structured_text)

    # Step 5: RAG retrieval with correct season
    season = window_stats.get("season", "Unknown")
    rag_query = (
        f"{season} season month {month} "
        f"daytime output {window_stats['daytime_avg_kw']:.0f} kW "
        f"deviation {window_stats['residual_mean']:.0f} kW"
    )
    rag_context = get_rag_context(rag_query, hybrid_retriever, top_k=3)

    # Step 6: Get seasonal expected output
    expected_output_kw = seasonal_lookup.get(month, 2000.0)

    # Step 7: Build updated prompt
    prompt = build_llmsense_prompt_with_rag(
        local_summary, window_stats, zone_name,
        rag_context, expected_output_kw
    )

    # Step 8: Llama 3.3 70B reasoning
    reasoning = get_cloud_reasoning(prompt)

    result = {
        "zone": zone_name,
        "window_start": window_stats["window_start"],
        "window_end": window_stats["window_end"],
        "actual_output_kw": window_stats["daytime_avg_kw"],
        "expected_output_kw": expected_output_kw,
        "season": season,
        "local_summary": local_summary,
        "rag_query": rag_query,
        "reasoning": reasoning
    }

    return result


# ── Test same 3 windows ────────────────────────────────────────
# for idx in [0, 90, 180]:
#     result = run_pipeline_with_rag("Zone_A", idx)
#     if result:
#         print("=" * 60)
#         print(f"Zone: {result['zone']} | Season: {result['season']}")
#         print(f"Window: {result['window_start']} → {result['window_end']}")
#         print(f"Actual: {result['actual_output_kw']} kW | Expected: {result['expected_output_kw']} kW")
#         print()
#         print("── Reasoning ──")
#         print(result["reasoning"])
#         print("=" * 60)
#         print()

In [49]:
from time_series import compute_window_summaries, compute_hourly_average
import pandas as pd

all_zone_summaries = []

for zone_name in ["Zone_A", "Zone_B", "Zone_C", "Zone_D"]:
    print(f"\nProcessing {zone_name}...")
    
    # Get full 24-hour data for this zone (needed for FFT + residual)
    zone_full = combined_with_zenith[
        combined_with_zenith["zone"] == zone_name
    ].copy()
    
    # Recompute ac_power_kw on full dataset for this zone
    from pvlib import temperature, pvsystem
    zone_full["time"] = pd.to_datetime(zone_full["time"])
    
    zone_full["cell_temperature"] = temperature.faiman(
        poa_global=zone_full["global_tilted_irradiance"],
        temp_air=zone_full["temperature_2m"],
        wind_speed=zone_full["wind_speed_10m"],
        u0=U0, u1=U1
    )
    zone_full["dc_power_kw"] = pvsystem.pvwatts_dc(
        g_poa_effective=zone_full["global_tilted_irradiance"],
        temp_cell=zone_full["cell_temperature"],
        pdc0=PDC0,
        gamma_pdc=GAMMA_PDC
    ) / 1000
    zone_full["ac_power_kw"] = (
        zone_full["dc_power_kw"] * INVERTER_EFFICIENCY * (1 - SYSTEM_LOSSES)
    ).clip(lower=0)
    
    # Compute hourly average for this zone
    zone_full["hour"] = zone_full["time"].dt.hour
    hourly_avg = zone_full.groupby("hour")["ac_power_kw"].mean()
    
    # Generate window summaries
    summaries = compute_window_summaries(zone_full, hourly_avg, zone_name)
    all_zone_summaries.append(summaries)
    print(f"{zone_name}: {len(summaries)} windows generated")

# Combine all zones
all_zones_summaries_df = pd.concat(all_zone_summaries, ignore_index=True)
print(f"\nTotal windows across all zones: {len(all_zones_summaries_df)}")

# Save
all_zones_summaries_df.to_csv("./data/all_zones_window_summaries.csv", index=False)
print("Saved -> ./data/all_zones_window_summaries.csv")


Processing Zone_A...


C:\Users\divya\AppData\Local\Temp\ipykernel_54672\1100449329.py:24: pvlibDeprecationWarning: Parameter 'g_poa_effective' has been renamed since 0.13.0. and will be removed soon. Please use 'effective_irradiance' instead.
  zone_full["dc_power_kw"] = pvsystem.pvwatts_dc(


Generated 1820 window summaries for Zone_A
Zone_A: 1820 windows generated

Processing Zone_B...


C:\Users\divya\AppData\Local\Temp\ipykernel_54672\1100449329.py:24: pvlibDeprecationWarning: Parameter 'g_poa_effective' has been renamed since 0.13.0. and will be removed soon. Please use 'effective_irradiance' instead.
  zone_full["dc_power_kw"] = pvsystem.pvwatts_dc(


Generated 1820 window summaries for Zone_B
Zone_B: 1820 windows generated

Processing Zone_C...


C:\Users\divya\AppData\Local\Temp\ipykernel_54672\1100449329.py:24: pvlibDeprecationWarning: Parameter 'g_poa_effective' has been renamed since 0.13.0. and will be removed soon. Please use 'effective_irradiance' instead.
  zone_full["dc_power_kw"] = pvsystem.pvwatts_dc(


Generated 1820 window summaries for Zone_C
Zone_C: 1820 windows generated

Processing Zone_D...


C:\Users\divya\AppData\Local\Temp\ipykernel_54672\1100449329.py:24: pvlibDeprecationWarning: Parameter 'g_poa_effective' has been renamed since 0.13.0. and will be removed soon. Please use 'effective_irradiance' instead.
  zone_full["dc_power_kw"] = pvsystem.pvwatts_dc(


Generated 1820 window summaries for Zone_D
Zone_D: 1820 windows generated

Total windows across all zones: 7280
Saved -> ./data/all_zones_window_summaries.csv


In [50]:
from rag import setup_rag, load_embeddings, get_rag_context

# Load all zones window summaries (not just Zone_A)
all_zones_summaries_df = pd.read_csv("./data/all_zones_window_summaries.csv")

print(f"Total windows: {len(all_zones_summaries_df)}")
print(f"Zones: {all_zones_summaries_df['zone'].unique()}")
print(f"Date range: {all_zones_summaries_df['window_start'].min()} → {all_zones_summaries_df['window_start'].max()}")

# Rebuild RAG index with all zones
embeddings = load_embeddings()
hybrid_retriever, seasonal_lookup, _ = setup_rag(
    all_zones_summaries_df,
    embeddings=embeddings,
    load_existing=False  # rebuild from scratch with all zones
)

print("\nRAG index rebuilt with all 4 zones.")

# Quick test — retrieve something zone-specific
test_query = "Zone_B monsoon July low output"
context = get_rag_context(test_query, hybrid_retriever)
print(f"\nTest query: '{test_query}'")
print(context)

Total windows: 7280
Zones: <ArrowStringArray>
['Zone_A', 'Zone_B', 'Zone_C', 'Zone_D']
Length: 4, dtype: str
Date range: 2020-01-01 00:00:00+05:30 → 2024-12-24 00:00:00+05:30
Loading embeddings: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7356.92it/s]


Embeddings loaded.
Created 7280 window summary documents.
Created 12 seasonal baseline documents.
Total documents: 7292
Building FAISS index from 7292 documents...
FAISS index built.
FAISS index saved -> ./data/faiss_index
Hybrid retriever ready.

RAG index rebuilt with all 4 zones.

Test query: 'Zone_B monsoon July low output'
[1] (window_summary | Monsoon)
Window: 2021-08-18 00:00:00+05:30 to 2021-08-24 23:00:00+05:30. Season: Monsoon. Dominant cycle: 24.0 hours. Daytime average output: 1937.61 kW. Residual std deviation: 309.94 kW. Max deviation from typical: 1695.16 kW. Residual mean vs seasonal norm: -64.5 kW.

[2] (seasonal_baseline | Monsoon)
Seasonal Baseline for July (Monsoon season). Based on 5 years of historical data (2020-2024). Typical daytime average output: 1797.4 kW. Output range: 954.4 to 2143.0 kW. Typical residual std deviation: 356.3 kW. Typical max deviation from norm: 1854.0 kW. Plant nameplate capacity: 5000 kW per zone. Grid contract minimum: 1500 kW during day

In [52]:
from config import SEASON_MAP

# Update summaries_df to use all zones
summaries_df = all_zones_summaries_df.copy()
summaries_df["month"] = pd.to_datetime(
    summaries_df["window_start"]
).dt.month
summaries_df["season"] = summaries_df["month"].map(SEASON_MAP)

print(f"summaries_df updated: {len(summaries_df)} windows")
print(summaries_df["zone"].value_counts())

summaries_df updated: 7280 windows
zone
Zone_A    1820
Zone_B    1820
Zone_C    1820
Zone_D    1820
Name: count, dtype: int64
